# 02b. Aplicar Splink (predict + clustering)

Carrega `splink_model.json` treinado no [`02_treinar_splink.ipynb`](02_treinar_splink.ipynb).
Não retreina: se o JSON não existir, falha apontando o notebook de treino.

`predict(0.5)` só gera candidatos. Clustering usa `THRESHOLD_AVALIACAO`
(env, default 0,95).

Próximo: [`03_avaliar.ipynb`](03_avaliar.ipynb) (score/cluster) e
[`04_atribuir.ipynb`](04_atribuir.ipynb) (1 CPF por Censo).


In [ ]:
import json
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from IPython.display import display
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
if not SPLINK_MODEL_JSON.exists():
    raise RuntimeError(
        f'Modelo Splink não encontrado: {SPLINK_MODEL_JSON}. '
        'Rode notebooks/02_treinar_splink.ipynb para treinar e gravar o JSON. '
        'Este notebook só carrega o modelo — não retreina.'
    )

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)
print('Modelo:', SPLINK_MODEL_JSON)
print('Threshold (clusters):', THRESHOLD_AVALIACAO)


## Linker a partir do JSON

Reconstrói o `Linker` como o 03: `json.load` + `Linker(..., data, db_api=...)`.
Comparisons vêm do JSON; não se redefinem aqui.


In [ ]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
print(
    'splink_censo:', con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0],
    '| splink_cpf:', con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0],
)

with open(SPLINK_MODEL_JSON, 'r') as file:
    data = json.load(file)

data['retain_intermediate_calculation_columns'] = True

linker = Linker(
    ['splink_censo', 'splink_cpf'],
    data,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


## Predict + clustering

`predict(0.5)` só gera candidatos. Clustering e validação usam
`THRESHOLD_AVALIACAO` (env, default 0,95).


In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = df_predict.as_pandas_dataframe()


In [ ]:
print('Threshold (clusters):', THRESHOLD_AVALIACAO)
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=THRESHOLD_AVALIACAO,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())


## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).


In [ ]:
records_to_plot = df_predictions.tail(5).to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)


In [ ]:
pd.set_option('display.max_columns', None)
df_predictions[
    (df_predictions['unique_id_l'].str.contains("censo"))
    & df_predictions['unique_id_r'].str.contains("cpf")
].head(5)


## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).


In [ ]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))


## Diagnóstico dos scores (sem rótulos)

Distribuição de match weight e tamanho de cluster. Avaliação no
[`03_avaliar.ipynb`](03_avaliar.ipynb); lista 1 CPF por Censo no
[`04_atribuir.ipynb`](04_atribuir.ipynb). Cheque os match weights: discordar nome/DOB
deve penalizar vários bits, não ≈ 0.


In [ ]:
display(df_predictions['match_probability'].describe())
faixas = pd.cut(df_predictions['match_weight'], bins=20)
display(
    df_predictions.groupby(faixas, observed=True)
    .size()
    .rename('n_pares')
    .to_frame()
)


In [ ]:
tamanhos = df_clusters.groupby('cluster_id').size()
print(f'Clusters: {tamanhos.size:,} | maior: {tamanhos.max():,} | singletons: {(tamanhos == 1).sum():,}')
display(
    tamanhos.value_counts().sort_index().head(20)
    .rename('n_clusters').to_frame().rename_axis('tamanho')
)

# Clusters grandes demais indicam blocking/threshold frouxo — inspecionar antes do 03/04.
display(tamanhos.sort_values(ascending=False).head(10).rename('tamanho').to_frame())


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(SPLINK_PREDICTIONS)
df_clusters.to_parquet(SPLINK_CLUSTERS)
print('Predictions:', SPLINK_PREDICTIONS)
print('Clusters:', SPLINK_CLUSTERS)


## Encerrar

Artefatos prontos para o [`03_avaliar.ipynb`](03_avaliar.ipynb) e o
[`04_atribuir.ipynb`](04_atribuir.ipynb).


In [ ]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
    ('clusters', SPLINK_CLUSTERS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')


In [ ]:
con.close()
